In [52]:
import pdfplumber
import pandas as pd
import re

# ------------------------
# 1. Normalizar texto
# ------------------------
def normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def parse_numeric(val):
    if not val:
        return 0
    val = str(val).replace("$","").replace(",","").strip()
    try:
        return float(val)
    except:
        return 0

# ------------------------
# 2. Obtener headers
# ------------------------
def get_headers(words):
    reg_word = next((w for w in words if w["text"].strip().lower() == "reg."), None)
    if not reg_word:
        return [], 0
    header_y = reg_word["top"]
    line_words = [w for w in words if abs(w["top"] - header_y) < 5]
    line_words = sorted(line_words, key=lambda w: w["x0"])
    headers = []
    for w in line_words:
        txt = w["text"].replace("-", "").strip()
        if txt.lower().startswith("reg"):
            name = "Reg."
        elif txt.lower() in ("c", "co", "c.o"):
            name = "C. O."
        elif txt.lower().startswith("doc"):
            name = "Doc. Cruce"
        elif txt.lower().startswith("detalle"):
            name = "Detalle"
        elif txt.lower().startswith("descuen"):
            name = "Descuentos"
        elif txt.lower().startswith("reten"):
            name = "Retenciones"
        elif txt.lower().startswith("valor fac"):
            name = "Valor Factura"
        elif txt.lower().startswith("valor pago") or txt.lower() == "pago":
            name = "Valor Pago"
        else:
            name = txt
        headers.append((name, w["x0"]))
    headers = sorted(headers, key=lambda x: x[1])
    return [h[0] for h in headers], [h[1] for h in headers]

# ------------------------
# 3. Extraer tabla
# ------------------------
def extract_pdf_table(path):
    rows_all = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            if not words:
                continue

            headers, xs = get_headers(words)
            if not headers:
                continue

            # cada columna: inicio x0 = header, fin = siguiente header -1
            col_ranges = []
            for i, x0 in enumerate(xs):
                x1 = xs[i+1]-1 if i+1 < len(xs) else 9999
                col_ranges.append((x0, x1))

            header_y = min(w["top"] for w in words if w["text"].lower() == "reg.")
            data_words = [w for w in words if w["top"] > header_y + 5]

            # agrupar palabras por línea
            lines = {}
            for w in data_words:
                y = round(w["top"])
                lines.setdefault(y, []).append(w)

            buffer = None
            for y, ws in sorted(lines.items()):
                row = {c: "" for c in headers}
                for w in ws:
                    x = w["x0"]
                    assigned = False
                    for i, (start, end) in enumerate(col_ranges):
                        if start <= x < end:
                            row[headers[i]] += " " + w["text"]
                            assigned = True
                            break
                    if not assigned:
                        # si no cae en ningún rango, última columna
                        row[headers[-1]] += " " + w["text"]
                row = {k: normalize_whitespace(v) for k,v in row.items()}

                # detectar nueva fila
                is_new = bool(re.match(r"^\d+", row.get("Reg.", ""))) or bool(row.get("C. O.", ""))
                if is_new:
                    if buffer:
                        rows_all.append(buffer)
                    buffer = row
                else:
                    if buffer:
                        buffer["Detalle"] = normalize_whitespace(buffer.get("Detalle","") + " " + row.get("Detalle",""))
                        for c in ["Descuentos","Retenciones","Valor Factura","Valor Pago"]:
                            if row.get(c):
                                buffer[c] = row[c]
                    else:
                        buffer = row
            if buffer:
                rows_all.append(buffer)

    df = pd.DataFrame(rows_all)

    # limpiar valores numéricos
    for c in ["Descuentos","Retenciones","Valor Factura","Valor Pago"]:
        if c in df.columns:
            df[c] = df[c].apply(parse_numeric)

    # columnas consistentes
    final_cols = ["Reg.","C. O.","Doc. Cruce","Detalle","Descuentos","Retenciones","Valor Factura","Valor Pago"]
    df = df[[c for c in final_cols if c in df.columns]]
    df = df.reset_index(drop=True)
    return df

# ------------------------
# 4. Leer PDFs
# ------------------------
df1 = extract_pdf_table("Remittance_euro.pdf")
df2 = extract_pdf_table("Remittance_euro_2.pdf")


(   Reg. Doc. Cruce                                            Detalle  \
 0                                    DETERGENTES 7 DE POS DEL OC- POS   
 1     1             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 2     2               POS LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19   
 3                   2025/EXHIBI- NOVIEM- DETERGENTES 7 DE POS DEL ...   
 4     3             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 5     4             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 6     5             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 7     6                        OC- POS LIQUIDOS DE OCTUBRE POS AHORRO   
 8                   DEL 15 AL 19 2025/EXHIBI- NOVIEM- DETERGENTES ...   
 9     7             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 10    8             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 11    9             LIQUIDOS DE OCTUBRE POS AHORRO DEL 15 AL 19 20...   
 12   10                              

In [58]:
df1, df2

,C. O.
0,ADM\nADM
1,
